# <font color="steelblue">Despliegue de modelos de aprendizaje automático</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**


**Fecha última edición**: 10/06/2026

**Licencia**: <small><a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a><br /></small>

No olvides hacer una copia si deseas utilizarlo. Al usar estos contenidos, aceptas nuestros términos de uso y nuestra política de privacidad.

In [ ]:
# @title Cargar módulos
# Cargamos módulos de análisis numérico
import numpy as np          # importamos numpy como np
import pandas as pd         # importamos pandas como pd
import math
import random                 # importamos módulo para cáculos matemáticos
from io import StringIO
import sys

# Cargamos módulos de análisis gráficos
from plotnine import *      # importamos módulo para gráficos con ggplot
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid') # Apply a seaborn style with a white background

%config InlineBackend.figure_format = 'retina'

# **1. Introducción y comparativa**

## 1.1 ¿Por qué necesitamos estas herramientas?

Los notebooks de Jupyter y Colab son ideales para exploración y desarrollo, pero tienen una limitación importante: **no son accesibles para usuarios no técnicos**. Para compartir un análisis o un modelo con un equipo de negocio, un cliente o el público general, necesitamos interfaces web.

Streamlit, Gradio y Dash resuelven este problema de formas distintas y complementarias:

| Característica | Streamlit | Gradio | Dash |
|---|---|---|---|
| **Curva de aprendizaje** | Muy baja | Mínima | Media |
| **Ideal para** | Apps de datos completas | Demos de modelos ML | Dashboards analíticos |
| **Personalización** | Media | Baja-Media | Alta |
| **Compartir sin servidor** | Streamlit Cloud (gratis) | Hugging Face Spaces (gratis) | Requiere servidor |
| **Desde Colab** | ngrok / localtunnel | Link automático integrado | ngrok / localtunnel |
| **Interactividad** | Script completo se re-ejecuta | Función específica se ejecuta | Callbacks reactivos |
| **Empresa detrás** | Snowflake | Hugging Face | Plotly |
| **Licencia** | Apache 2.0 | Apache 2.0 | MIT |

## 1.2 ¿Cuándo usar cada uno?

```
¿Necesitas demostrar un modelo de ML rápidamente?
  └── GRADIO → link público inmediato, sin configuración

¿Necesitas una app de datos con filtros, tablas y gráficos?
  └── STREAMLIT → sintaxis simple, deploy fácil en Streamlit Cloud

¿Necesitas un dashboard profesional con múltiples páginas,
   callbacks complejos y máximo control visual?
  └── DASH → más código pero más potente y personalizable
```

# **2. Configuración del entorno en Colab**


## 2.1 Instalación de librerías

Antes de escribir ninguna app, necesitamos instalar los tres frameworks y sus dependencias. También instalamos `pyngrok`, que será la pieza clave para hacer accesibles desde el exterior las apps de Streamlit y Dash: actúa como un túnel que conecta un puerto local de la máquina virtual de Colab con una URL pública en internet. Gradio no necesita esto porque genera su propio enlace interno. El bloque de verificación al final confirma que todas las librerías están disponibles antes de continuar.

In [ ]:
!pip install streamlit gradio dash dash-bootstrap-components \
             pyngrok plotly  \
             xgboost --quiet

# Verificar instalaciones
import importlib
for lib in ['streamlit', 'gradio', 'dash', 'plotly', 'sklearn', 'xgboost']:
    try:
        m = importlib.import_module(lib)
        v = getattr(m, '__version__', '?')
        print(f"  ✓ {lib} {v}")
    except ImportError:
        print(f"  ✗ {lib} — error en instalación")

## 2.2 Preparar datos y modelos de ejemplo

A lo largo del cuaderno usaremos tres datasets estándar de scikit-learn que ya conocemos del módulo anterior: **Breast Cancer** para clasificación binaria (maligno/benigno), **California Housing** para regresión (precio de viviendas) e **Iris** para clasificación multiclase. Los entrenamos aquí una sola vez y los reutilizamos en las tres secciones de frameworks. Esto nos permite centrarnos en la interfaz de cada herramienta sin repetir código de ML en cada bloque.



In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer, load_iris, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# ── Dataset 1: Breast Cancer (clasificación) ──────────────────────────────────
bc = load_breast_cancer()
X_bc = pd.DataFrame(bc.data, columns=bc.feature_names)
y_bc = pd.Series(bc.target, name='target')

X_tr_bc, X_te_bc, y_tr_bc, y_te_bc = train_test_split(
    X_bc, y_bc, test_size=0.25, random_state=42, stratify=y_bc)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_tr_bc, y_tr_bc)
acc_bc = accuracy_score(y_te_bc, clf.predict(X_te_bc))
print(f"✓ Clasificador Breast Cancer — Accuracy: {acc_bc:.4f}")

# ── Dataset 2: California Housing (regresión) ──────────────────────────────────
housing = fetch_california_housing()
X_hs = pd.DataFrame(housing.data, columns=housing.feature_names)
y_hs = pd.Series(housing.target, name='MedHouseVal')
df_housing = pd.concat([X_hs, y_hs], axis=1)

X_tr_hs, X_te_hs, y_tr_hs, y_te_hs = train_test_split(
    X_hs, y_hs, test_size=0.2, random_state=42)

reg = GradientBoostingRegressor(n_estimators=100, random_state=42)
reg.fit(X_tr_hs, y_tr_hs)
rmse_hs = mean_squared_error(y_te_hs, reg.predict(X_te_hs), squared=False)
print(f"✓ Regresor California Housing — RMSE: {rmse_hs:.4f}")

# ── Dataset 3: Iris (para Gradio) ──────────────────────────────────────────────
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = pd.Series(iris.target)

clf_iris = RandomForestClassifier(n_estimators=50, random_state=42)
clf_iris.fit(X_iris, y_iris)
print(f"✓ Clasificador Iris — Accuracy: {accuracy_score(y_iris, clf_iris.predict(X_iris)):.4f}")

print("\nTodos los modelos y datasets preparados.")

## 2.3 Configurar ngrok para túnel público

Google Colab ejecuta código en una máquina virtual sin dirección IP pública. Para que alguien externo pueda acceder a una app que corre en esa máquina, necesitamos un **túnel inverso**: ngrok abre una conexión desde la máquina de Colab hacia sus servidores, y expone esa conexión como una URL pública del tipo `https://xxxx.ngrok.io`. El token de autenticación es gratuito y se obtiene en [dashboard.ngrok.com](https://dashboard.ngrok.com) con solo registrarse. Sin él, la cuenta anónima solo permite un túnel simultáneo con sesiones muy cortas. La función `abrir_tunel` que definimos aquí la llamaremos en las secciones de Streamlit y Dash cada vez que queramos publicar una app.

In [ ]:
# ngrok permite exponer un puerto local de Colab como URL pública
# Necesitas una cuenta gratuita en https://ngrok.com y un authtoken

from pyngrok import ngrok, conf
import os

# ── OPCIÓN A: pegar tu authtoken aquí ─────────────────────────────────────────
# Obtén tu token gratis en: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ""   # ← pega tu token aquí

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✓ ngrok configurado con authtoken")
else:
    print("⚠️  Sin authtoken de ngrok.")
    print("   Las apps de Streamlit y Dash necesitan ngrok para ser accesibles.")
    print("   Gradio genera su propio link sin necesidad de ngrok.")
    print("   Obtén tu token gratuito en: https://dashboard.ngrok.com")

# ── Función helper para abrir un túnel ────────────────────────────────────────
def abrir_tunel(puerto, tipo="http"):
    """Abre un túnel ngrok y devuelve la URL pública."""
    try:
        tunel = ngrok.connect(puerto, tipo)
        url = tunel.public_url
        print(f"✓ Túnel abierto: {url} → localhost:{puerto}")
        return url
    except Exception as e:
        print(f"✗ Error al abrir túnel: {e}")
        return None

def cerrar_tuneles():
    """Cierra todos los túneles ngrok abiertos."""
    ngrok.kill()
    print("Todos los túneles cerrados.")

# **3. Streamlit — Análisis de DataFrames**

## 3.1 ¿Cómo funciona Streamlit en Colab?

Streamlit está diseñado para ejecutarse como proceso independiente, no dentro de un notebook. En Colab, el flujo es:

```
1. Escribir el código de la app en un archivo .py  (con %%writefile)
2. Lanzar streamlit run app.py en segundo plano
3. Abrir un túnel ngrok hacia el puerto 8501
4. Acceder a la URL pública del túnel
```

## 3.2 App 1: Explorador de DataFrames

La primera app que construimos con Streamlit es un explorador genérico de datasets: puede cargar cualquier CSV que el usuario suba o trabajar con los datasets de ejemplo. La app se organiza en seis secciones que reflejan un flujo típico de análisis exploratorio: resumen de dimensiones, previsualización de filas, estadísticas descriptivas, análisis de nulos, visualizaciones interactivas (histograma y scatter con Plotly) y filtros por rango. Al final, el usuario puede descargar los datos filtrados como CSV directamente desde la interfaz. Fíjate en el uso de `@st.cache_data` para que la carga del dataset no se repita en cada interacción, y en `st.session_state` para persistir el estado entre rerenderizados.

In [ ]:
%%writefile streamlit_explorador.py
"""
Streamlit App 1: Explorador interactivo de DataFrames
Permite cargar, filtrar, visualizar y resumir cualquier CSV.
"""

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from io import StringIO

# ── Configuración de la página ────────────────────────────────────────────────
st.set_page_config(
    page_title="Explorador de Datos",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── Estilos personalizados ─────────────────────────────────────────────────────
st.markdown("""
<style>
    .metric-card {
        background: #f0f4ff;
        border: 1px solid #c7d7fd;
        border-radius: 8px;
        padding: 16px;
        text-align: center;
    }
    .stDataFrame { font-size: 13px; }
    h1 { color: #1e3a8a; }
    h2 { color: #1e40af; border-bottom: 2px solid #bfdbfe; padding-bottom: 6px; }
</style>
""", unsafe_allow_html=True)

# ── Título y descripción ───────────────────────────────────────────────────────
st.title("📊 Explorador Interactivo de DataFrames")
st.markdown("Carga cualquier CSV, explora sus estadísticas y genera visualizaciones.")

# ── Sidebar: carga de datos ────────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Configuración")
    fuente = st.radio("Fuente de datos:", ["Dataset de ejemplo", "Subir CSV"])

    if fuente == "Dataset de ejemplo":
        dataset_nombre = st.selectbox(
            "Selecciona un dataset:",
            ["California Housing", "Iris", "Breast Cancer (muestra)"]
        )
    else:
        archivo = st.file_uploader("Sube tu CSV", type=['csv'])

    st.divider()
    st.markdown("**Opciones de visualización:**")
    mostrar_nulos = st.checkbox("Mostrar análisis de nulos", True)
    mostrar_corr  = st.checkbox("Mostrar mapa de correlación", True)
    n_filas_prev  = st.slider("Filas en previsualización", 5, 50, 10)

# ── Cargar datos ───────────────────────────────────────────────────────────────
@st.cache_data
def cargar_dataset(nombre):
    from sklearn.datasets import (load_iris, load_breast_cancer,
                                   fetch_california_housing)
    if nombre == "California Housing":
        d = fetch_california_housing()
        df = pd.DataFrame(d.data, columns=d.feature_names)
        df['MedHouseVal'] = d.target
    elif nombre == "Iris":
        d = load_iris()
        df = pd.DataFrame(d.data, columns=d.feature_names)
        df['species'] = [d.target_names[i] for i in d.target]
    else:
        d = load_breast_cancer()
        df = pd.DataFrame(d.data[:200], columns=d.feature_names)
        df['diagnosis'] = ['malignant' if t == 0 else 'benign' for t in d.target[:200]]
    return df

try:
    if fuente == "Dataset de ejemplo":
        df = cargar_dataset(dataset_nombre)
    else:
        if archivo is not None:
            df = pd.read_csv(archivo)
        else:
            st.info("👆 Sube un archivo CSV para comenzar.")
            st.stop()
except Exception as e:
    st.error(f"Error al cargar datos: {e}")
    st.stop()

# ── Métricas resumen ───────────────────────────────────────────────────────────
st.header("1. Resumen del Dataset")
col1, col2, col3, col4 = st.columns(4)
col1.metric("📋 Filas",       f"{df.shape[0]:,}")
col2.metric("📐 Columnas",    f"{df.shape[1]:,}")
col3.metric("🔢 Numéricas",   f"{df.select_dtypes(include='number').shape[1]}")
col4.metric("⚠️ Nulos totales", f"{df.isnull().sum().sum():,}")

# ── Previsualización ───────────────────────────────────────────────────────────
with st.expander("🔍 Previsualización de datos", expanded=True):
    tab1, tab2 = st.tabs(["Primeras filas", "Últimas filas"])
    with tab1:
        st.dataframe(df.head(n_filas_prev), use_container_width=True)
    with tab2:
        st.dataframe(df.tail(n_filas_prev), use_container_width=True)

# ── Estadísticas descriptivas ─────────────────────────────────────────────────
st.header("2. Estadísticas Descriptivas")
cols_num = df.select_dtypes(include='number').columns.tolist()
if cols_num:
    st.dataframe(
        df[cols_num].describe().round(4).style.background_gradient(
            cmap='Blues', axis=1),
        use_container_width=True
    )

# ── Análisis de nulos ──────────────────────────────────────────────────────────
if mostrar_nulos:
    st.header("3. Análisis de Valores Nulos")
    nulos = df.isnull().sum()
    nulos_pct = (nulos / len(df) * 100).round(2)
    df_nulos = pd.DataFrame({'Nulos': nulos, '% del total': nulos_pct})
    df_nulos = df_nulos[df_nulos['Nulos'] > 0]

    if len(df_nulos) > 0:
        st.warning(f"Se encontraron {df_nulos['Nulos'].sum()} valores nulos en "
                   f"{len(df_nulos)} columnas.")
        col_n1, col_n2 = st.columns([1, 2])
        with col_n1:
            st.dataframe(df_nulos, use_container_width=True)
        with col_n2:
            fig_nulos = px.bar(
                df_nulos.reset_index(), x='index', y='% del total',
                title='% de nulos por columna',
                color='% del total', color_continuous_scale='Reds'
            )
            st.plotly_chart(fig_nulos, use_container_width=True)
    else:
        st.success("✓ No hay valores nulos en el dataset.")

# ── Visualizaciones ────────────────────────────────────────────────────────────
st.header("4. Visualizaciones")
col_v1, col_v2 = st.columns(2)

with col_v1:
    var_hist = st.selectbox("Variable para histograma:", cols_num)
    if var_hist:
        fig_hist = px.histogram(
            df, x=var_hist, nbins=30,
            title=f"Distribución de {var_hist}",
            color_discrete_sequence=['#2563eb']
        )
        fig_hist.update_layout(bargap=0.05)
        st.plotly_chart(fig_hist, use_container_width=True)

with col_v2:
    if len(cols_num) >= 2:
        var_x = st.selectbox("Eje X (scatter):", cols_num, index=0)
        var_y = st.selectbox("Eje Y (scatter):", cols_num, index=min(1, len(cols_num)-1))
        cols_cat = df.select_dtypes(exclude='number').columns.tolist()
        color_var = st.selectbox("Color por:", ['Ninguno'] + cols_cat + cols_num)
        color_arg = None if color_var == 'Ninguno' else color_var
        fig_scatter = px.scatter(
            df.sample(min(500, len(df))), x=var_x, y=var_y,
            color=color_arg, title=f"{var_x} vs {var_y}",
            opacity=0.7
        )
        st.plotly_chart(fig_scatter, use_container_width=True)

# ── Mapa de correlación ────────────────────────────────────────────────────────
if mostrar_corr and len(cols_num) > 1:
    st.header("5. Mapa de Correlación")
    vars_corr = st.multiselect(
        "Variables a incluir:",
        cols_num,
        default=cols_num[:min(10, len(cols_num))]
    )
    if len(vars_corr) >= 2:
        corr_matrix = df[vars_corr].corr().round(3)
        fig_corr = px.imshow(
            corr_matrix,
            color_continuous_scale='RdBu_r',
            zmin=-1, zmax=1,
            title="Mapa de correlación de Pearson",
            text_auto=True
        )
        fig_corr.update_layout(height=500)
        st.plotly_chart(fig_corr, use_container_width=True)

# ── Filtros interactivos ───────────────────────────────────────────────────────
st.header("6. Filtros Interactivos")
with st.expander("🔧 Configurar filtros"):
    df_filtrado = df.copy()
    cols_filtro = st.multiselect("Columnas a filtrar:", cols_num)
    for col in cols_filtro:
        min_v, max_v = float(df[col].min()), float(df[col].max())
        rango = st.slider(f"Rango de {col}:", min_v, max_v, (min_v, max_v))
        df_filtrado = df_filtrado[
            (df_filtrado[col] >= rango[0]) & (df_filtrado[col] <= rango[1])
        ]

    st.write(f"**{len(df_filtrado):,} filas** tras los filtros "
             f"({len(df_filtrado)/len(df):.1%} del total)")
    st.dataframe(df_filtrado.head(20), use_container_width=True)

    # Descargar datos filtrados
    csv_filtrado = df_filtrado.to_csv(index=False).encode('utf-8')
    st.download_button(
        "⬇️ Descargar datos filtrados (CSV)",
        data=csv_filtrado,
        file_name="datos_filtrados.csv",
        mime="text/csv"
    )

st.divider()
st.caption("Explorador de DataFrames · Curso de Machine Learning 2025")

## 3.3 Lanzar la app Streamlit desde Colab

Como Streamlit no puede correr dentro de una celda de Jupyter, necesitamos lanzarlo como proceso independiente en segundo plano con `subprocess.Popen`. La función `lanzar_streamlit` encapsula toda la lógica: mata procesos anteriores en el mismo puerto para evitar conflictos, arranca el servidor con las opciones necesarias para entorno headless (sin pantalla), espera a que inicialice y por último abre el túnel ngrok. El resultado es una URL pública que puedes abrir en cualquier navegador, en cualquier dispositivo, mientras la celda de Colab siga activa. Cuando termines, llama a `proc.terminate()` para liberar el puerto.

In [ ]:
import subprocess
import time
import threading

def lanzar_streamlit(archivo, puerto=8501):
    """
    Lanza Streamlit en segundo plano y abre un túnel ngrok.
    Devuelve la URL pública para acceder a la app.
    """
    # Matar procesos anteriores en el mismo puerto
    subprocess.run(['pkill', '-f', f'streamlit'], capture_output=True)
    time.sleep(1)

    # Lanzar Streamlit en segundo plano
    proceso = subprocess.Popen(
        ['streamlit', 'run', archivo,
         '--server.port', str(puerto),
         '--server.headless', 'true',
         '--server.enableCORS', 'false',
         '--server.enableXsrfProtection', 'false'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    # Esperar a que arranque
    time.sleep(4)

    if NGROK_TOKEN:
        url = abrir_tunel(puerto)
    else:
        url = f"http://localhost:{puerto}"
        print("⚠️  Sin ngrok: la app solo es accesible dentro de Colab.")
        print("   Configura NGROK_TOKEN para obtener una URL pública.")

    return proceso, url

# ── Lanzar el explorador de DataFrames ────────────────────────────────────────
proc_st1, url_st1 = lanzar_streamlit('streamlit_explorador.py', puerto=8501)
print(f"\n🚀 App Streamlit disponible en: {url_st1}")
print("   Abre la URL en tu navegador para ver la aplicación.")
print("   Para detenerla: proc_st1.terminate()")

## 3.4 App 2: Dashboard de modelo ML

La segunda app de Streamlit da un paso más: en lugar de explorar datos estáticos, permite al usuario **configurar y entrenar un modelo de ML en tiempo real** desde la interfaz. La barra lateral expone los hiperparámetros principales (tipo de modelo, `n_estimators`, `max_depth`, tamaño del test) y un botón de entrenamiento. Los resultados se muestran en cuatro pestañas: métricas globales y matriz de confusión, curva ROC con distribución de probabilidades, importancia de variables y, la más interactiva, un panel de predicción manual donde el usuario puede ajustar sliders para cada variable y ver la predicción actualizarse en tiempo real. Esta app ilustra dos patrones avanzados de Streamlit: el uso de `st.session_state` para recordar que el modelo ya fue entrenado entre rerenderizados, y el layout de pestañas con `st.tabs`.

In [ ]:
%%writefile streamlit_modelo_ml.py
"""
Streamlit App 2: Dashboard interactivo para análisis de modelos de ML.
Muestra métricas, importancia de variables, predicciones y errores.
"""

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_curve, auc)

st.set_page_config(page_title="ML Dashboard", page_icon="🤖",
                    layout="wide")

st.title("🤖 Dashboard de Análisis de Modelos ML")

# ── Sidebar ────────────────────────────────────────────────────────────────────
with st.sidebar:
    st.header("Configuración del Modelo")
    dataset_sel = st.selectbox("Dataset:", ["Breast Cancer"])
    modelo_sel  = st.selectbox("Modelo:", ["Random Forest", "Gradient Boosting"])
    n_estimators = st.slider("n_estimators:", 10, 300, 100, step=10)
    max_depth    = st.slider("max_depth:", 1, 20, 5)
    test_size    = st.slider("% test:", 0.1, 0.4, 0.25, step=0.05)
    entrenar_btn = st.button("🚀 Entrenar modelo", type="primary")

# ── Inicializar estado ─────────────────────────────────────────────────────────
if 'modelo_entrenado' not in st.session_state:
    st.session_state.modelo_entrenado = False

# ── Cargar datos ───────────────────────────────────────────────────────────────
@st.cache_data
def cargar_datos(nombre):
    d = load_breast_cancer()
    X = pd.DataFrame(d.data, columns=d.feature_names)
    y = pd.Series(d.target)
    class_names = list(d.target_names)
    return X, y, class_names

X, y, class_names = cargar_datos(dataset_sel)

# ── Entrenar al pulsar el botón ────────────────────────────────────────────────
if entrenar_btn or st.session_state.modelo_entrenado:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y)

    if modelo_sel == "Random Forest":
        modelo = RandomForestClassifier(n_estimators=n_estimators,
                                         max_depth=max_depth, random_state=42)
    else:
        modelo = GradientBoostingClassifier(n_estimators=n_estimators,
                                              max_depth=max_depth, random_state=42)

    with st.spinner("Entrenando modelo..."):
        modelo.fit(X_tr, y_tr)
        y_pred = modelo.predict(X_te)
        y_proba = modelo.predict_proba(X_te)[:, 1]
        acc = accuracy_score(y_te, y_pred)
        cv_scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')

    st.session_state.modelo_entrenado = True

    # ── Tab layout ────────────────────────────────────────────────────────────
    tab1, tab2, tab3, tab4 = st.tabs(
        ["📊 Métricas", "📈 ROC & Curvas", "🔍 Variables", "🔮 Predicciones"]
    )

    with tab1:
        st.subheader("Métricas de rendimiento")
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Accuracy (test)",  f"{acc:.4f}")
        c2.metric("CV media (5-fold)", f"{cv_scores.mean():.4f}")
        c3.metric("CV std",           f"{cv_scores.std():.4f}")
        c4.metric("Train size",       f"{len(X_tr)}")

        col_cm, col_rep = st.columns(2)
        with col_cm:
            cm = confusion_matrix(y_te, y_pred)
            fig_cm = px.imshow(
                cm, text_auto=True,
                x=class_names, y=class_names,
                color_continuous_scale='Blues',
                title="Matriz de Confusión"
            )
            fig_cm.update_layout(height=350)
            st.plotly_chart(fig_cm, use_container_width=True)

        with col_rep:
            report = classification_report(y_te, y_pred,
                                           target_names=class_names,
                                           output_dict=True)
            df_report = pd.DataFrame(report).T.round(3)
            st.dataframe(df_report.style.background_gradient(
                cmap='Greens', subset=['precision','recall','f1-score']),
                use_container_width=True)

    with tab2:
        fpr, tpr, _ = roc_curve(y_te, y_proba)
        auc_score = auc(fpr, tpr)
        fig_roc = go.Figure()
        fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                                      name=f'ROC (AUC={auc_score:.4f})',
                                      line=dict(color='#2563eb', width=2.5)))
        fig_roc.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                                      name='Azar', line=dict(dash='dash',
                                      color='gray', width=1)))
        fig_roc.update_layout(
            title=f'Curva ROC — AUC = {auc_score:.4f}',
            xaxis_title='Tasa de Falsos Positivos',
            yaxis_title='Tasa de Verdaderos Positivos',
            height=400
        )
        st.plotly_chart(fig_roc, use_container_width=True)

        # Distribución de probabilidades predichas
        df_proba = pd.DataFrame({'proba': y_proba, 'real': y_te.values})
        fig_dist = px.histogram(
            df_proba, x='proba', color='real',
            nbins=30, barmode='overlay', opacity=0.7,
            title='Distribución de probabilidades predichas por clase',
            color_discrete_map={0: '#E74C3C', 1: '#2ECC71'}
        )
        st.plotly_chart(fig_dist, use_container_width=True)

    with tab3:
        importancias = pd.DataFrame({
            'variable': X.columns,
            'importancia': modelo.feature_importances_
        }).sort_values('importancia', ascending=False)

        fig_imp = px.bar(
            importancias.head(15), x='importancia', y='variable',
            orientation='h', title='Top 15 variables más importantes',
            color='importancia', color_continuous_scale='Blues'
        )
        fig_imp.update_layout(yaxis={'categoryorder': 'total ascending'},
                               height=500)
        st.plotly_chart(fig_imp, use_container_width=True)

        st.dataframe(importancias.reset_index(drop=True), use_container_width=True)

    with tab4:
        st.subheader("Realizar predicción manual")
        st.markdown("Ajusta los valores de las variables para obtener una predicción:")
        cols_input = st.columns(3)
        input_vals = {}
        for i, col_name in enumerate(X.columns[:9]):
            with cols_input[i % 3]:
                min_v = float(X[col_name].min())
                max_v = float(X[col_name].max())
                mean_v = float(X[col_name].mean())
                input_vals[col_name] = st.slider(
                    col_name[:25], min_v, max_v, mean_v,
                    key=f"slider_{col_name}"
                )

        # Rellenar el resto con la media
        for col_name in X.columns[9:]:
            input_vals[col_name] = float(X[col_name].mean())

        X_input = pd.DataFrame([input_vals])
        pred_input = modelo.predict(X_input)[0]
        proba_input = modelo.predict_proba(X_input)[0]

        clase_pred = class_names[pred_input]
        color_pred = "🟢" if pred_input == 1 else "🔴"
        st.markdown(f"### {color_pred} Predicción: **{clase_pred}**")
        st.markdown(f"Probabilidad benigno: **{proba_input[1]:.4f}** | "
                    f"Probabilidad maligno: **{proba_input[0]:.4f}**")

else:
    st.info("👈 Configura el modelo en la barra lateral y pulsa **Entrenar modelo**.")

st.caption("ML Dashboard · Curso de Machine Learning 2025")

In [ ]:
# Detener app anterior si está corriendo
if 'proc_st1' in dir() and proc_st1:
    proc_st1.terminate()

proc_st2, url_st2 = lanzar_streamlit('streamlit_modelo_ml.py', puerto=8502)
print(f"\n🚀 ML Dashboard disponible en: {url_st2}")

# **4. Gradio — Interfaces para modelos de ML**


## 4.1 ¿Por qué Gradio?

Gradio tiene una ventaja única: **genera automáticamente un enlace público temporal** (válido 72 horas) sin necesidad de ngrok ni de ninguna configuración adicional. Es la opción más rápida para compartir un demo de un modelo. El código de Gradio vive directamente en las celdas del notebook, sin necesidad de `%%writefile` ni de lanzar un proceso separado: se define la función Python que hace la predicción, se le asocia una interfaz con `gr.Interface` o `gr.Blocks`, y se lanza con `.launch(share=True)`. El enlace aparece inmediatamente en la salida de la celda.

In [ ]:
import gradio as gr
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import plotly.express as px

## 4.2 App 1: Clasificador con explicación visual

Esta primera app de Gradio demuestra la forma más completa de construir una interfaz: usando `gr.Blocks`, que da control total sobre el layout, en lugar de `gr.Interface`, que genera automáticamente una interfaz de dos columnas. La función de predicción `predecir_iris` recibe cuatro medidas de la flor, llama al modelo y devuelve **tres salidas simultáneas**: un componente `gr.Label` con las probabilidades por clase, un gráfico matplotlib con las barras de confianza, y un DataFrame comparando los valores introducidos con las medias históricas de cada especie. Un detalle importante: conectamos el evento `slider.change` además del botón, de modo que la predicción se actualiza en tiempo real mientras el usuario mueve los sliders, sin necesidad de pulsar ningún botón.

In [ ]:
# ── Preparar el modelo Iris ────────────────────────────────────────────────────
iris = load_iris()
clf_iris = RandomForestClassifier(n_estimators=100, random_state=42)
clf_iris.fit(iris.data, iris.target)
nombres_iris = list(iris.target_names)
nombres_feat = [f.replace(' (cm)', '').replace(' ', '\n')
                for f in iris.feature_names]

def predecir_iris(sepal_length, sepal_width, petal_length, petal_width):
    """
    Predice la especie de Iris y devuelve:
    - Etiquetas de probabilidad para el componente Label
    - Un gráfico de barras con las probabilidades
    - Un DataFrame con la comparación con los datos de entrenamiento
    """
    X_in  = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    probs = clf_iris.predict_proba(X_in)[0]
    pred  = clf_iris.predict(X_in)[0]

    # ── Salida 1: diccionario para gr.Label ───────────────────────────────────
    label_dict = {nombre: float(p) for nombre, p in zip(nombres_iris, probs)}

    # ── Salida 2: gráfico de barras de probabilidades ─────────────────────────
    fig, ax = plt.subplots(figsize=(6, 3))
    colors = ['#2563eb' if i == pred else '#93c5fd' for i in range(3)]
    bars = ax.bar(nombres_iris, probs, color=colors, edgecolor='white',
                   linewidth=1.5)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Probabilidad')
    ax.set_title(f'Predicción: {nombres_iris[pred].upper()} '
                 f'(confianza: {probs[pred]:.1%})', fontweight='bold')
    for bar, p in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{p:.3f}', ha='center', fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()

    # ── Salida 3: tabla comparativa ────────────────────────────────────────────
    medias = pd.DataFrame(iris.data, columns=iris.feature_names).groupby(
        [iris.target_names[t] for t in iris.target]).mean().round(3)
    medias.index.name = 'Especie'
    fila_input = pd.DataFrame(
        [[sepal_length, sepal_width, petal_length, petal_width]],
        columns=iris.feature_names,
        index=[f'→ Tu muestra ({nombres_iris[pred]})']
    )
    tabla = pd.concat([fila_input, medias])

    return label_dict, fig, tabla

# ── Construir la interfaz ─────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title="Clasificador Iris") as demo_iris:
    gr.Markdown("""
    # 🌸 Clasificador de Iris
    Introduce las medidas de la flor y obtén la predicción del modelo
    con explicación visual.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Medidas de entrada")
            sepal_l = gr.Slider(4.0, 8.0, value=5.8, step=0.1,
                                 label="Sepal length (cm)")
            sepal_w = gr.Slider(2.0, 4.5, value=3.0, step=0.1,
                                 label="Sepal width (cm)")
            petal_l = gr.Slider(1.0, 7.0, value=3.8, step=0.1,
                                 label="Petal length (cm)")
            petal_w = gr.Slider(0.1, 2.5, value=1.2, step=0.1,
                                 label="Petal width (cm)")
            btn = gr.Button("🔍 Clasificar", variant="primary")

            # Ejemplos predefinidos
            gr.Examples(
                examples=[
                    [5.1, 3.5, 1.4, 0.2],   # setosa
                    [6.7, 3.1, 4.7, 1.5],   # versicolor
                    [6.3, 3.3, 6.0, 2.5],   # virginica
                ],
                inputs=[sepal_l, sepal_w, petal_l, petal_w],
                label="Ejemplos típicos"
            )

        with gr.Column(scale=2):
            gr.Markdown("### Resultados")
            with gr.Row():
                etiquetas  = gr.Label(num_top_classes=3, label="Probabilidades")
                grafico    = gr.Plot(label="Distribución de probabilidades")
            tabla_comp = gr.Dataframe(label="Comparación con medias por especie")

    btn.click(
        fn=predecir_iris,
        inputs=[sepal_l, sepal_w, petal_l, petal_w],
        outputs=[etiquetas, grafico, tabla_comp]
    )

    # Ejecutar al cambiar cualquier slider (interactividad en tiempo real)
    for slider in [sepal_l, sepal_w, petal_l, petal_w]:
        slider.change(
            fn=predecir_iris,
            inputs=[sepal_l, sepal_w, petal_l, petal_w],
            outputs=[etiquetas, grafico, tabla_comp]
        )

print("App Gradio definida. Ejecuta la siguiente celda para lanzarla.")

In [ ]:
# ── Lanzar la app y obtener URL pública ────────────────────────────────────────
# share=True genera automáticamente un link público gratuito (72h)
demo_iris.launch(
    share=True,          # ← URL pública automática, sin ngrok
    debug=False,
    quiet=True,
    show_error=True,
    inbrowser=False      # no abrir navegador en Colab
)

## 4.3 App 2: Análisis de CSV con IA

Esta segunda app ilustra un caso de uso muy frecuente en el trabajo diario: alguien sube un CSV, indica cuál es la columna objetivo y quiere obtener automáticamente un modelo entrenado con sus métricas y sus variables más importantes. La función `analizar_csv` maneja todo el pipeline de un golpe: lee el fichero, codifica las variables categóricas, imputa nulos con la mediana, divide en train/test, entrena el modelo elegido y devuelve el reporte de clasificación como texto Markdown, el gráfico de importancias y la distribución de clases. El objetivo es demostrar que Gradio puede ser una herramienta de análisis exploratorio rápido, no solo un escaparate para modelos ya entrenados.

In [ ]:
def analizar_csv(archivo, columna_objetivo, tipo_modelo):
    """
    Recibe un CSV, entrena un modelo y devuelve métricas y gráficos.
    """
    import io
    from sklearn.preprocessing import LabelEncoder
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, classification_report

    if archivo is None:
        return "⚠️ Sube un archivo CSV", None, None

    try:
        df = pd.read_csv(archivo.name)
    except Exception as e:
        return f"Error al leer CSV: {e}", None, None

    if columna_objetivo not in df.columns:
        cols = ', '.join(df.columns.tolist())
        return f"Columna '{columna_objetivo}' no encontrada.\nColumnas disponibles: {cols}", None, None

    # Preparar datos
    y = df[columna_objetivo]
    X = df.drop(columns=[columna_objetivo])

    # Codificar categóricas
    for col in X.select_dtypes(include='object').columns:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    if y.dtype == 'object':
        y = LabelEncoder().fit_transform(y)

    X = X.fillna(X.median())
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                                random_state=42)

    # Entrenar modelo
    if tipo_modelo == "Random Forest":
        model_csv = RandomForestClassifier(n_estimators=100, random_state=42)
    else:
        model_csv = LogisticRegression(max_iter=1000, random_state=42)
    model_csv.fit(X_tr, y_tr)
    y_pred = model_csv.predict(X_te)
    acc = accuracy_score(y_te, y_pred)

    # Texto de resultados
    reporte = f"""
## Resultados del análisis

**Dataset:** {df.shape[0]} filas × {df.shape[1]} columnas
**Modelo:** {tipo_modelo}
**Variable objetivo:** {columna_objetivo}
**Accuracy (test 25%):** {acc:.4f}

### Reporte de clasificación: {classification_report(y_te, y_pred)}
"""

    # Gráfico de importancias
    if hasattr(model_csv, 'feature_importances_'):
        imp_df = pd.DataFrame({
            'Variable': X.columns,
            'Importancia': model_csv.feature_importances_
        }).sort_values('Importancia', ascending=False).head(10)
        fig_imp, ax = plt.subplots(figsize=(8, 4))
        ax.barh(imp_df['Variable'], imp_df['Importancia'],
                color='#2563eb', alpha=0.85)
        ax.set_title('Top 10 variables más importantes', fontweight='bold')
        ax.set_xlabel('Importancia')
        ax.invert_yaxis()
        plt.tight_layout()
    else:
        fig_imp = None

    # Gráfico de distribución de la variable objetivo
    fig_dist, ax2 = plt.subplots(figsize=(6, 3))
    pd.Series(y).value_counts().plot(kind='bar', ax=ax2,
                                      color='#2563eb', edgecolor='white')
    ax2.set_title(f'Distribución de {columna_objetivo}', fontweight='bold')
    ax2.set_xlabel('Clase')
    ax2.set_ylabel('Frecuencia')
    plt.tight_layout()

    return reporte, fig_imp, fig_dist


with gr.Blocks(theme=gr.themes.Soft(), title="Analizador CSV") as demo_csv:
    gr.Markdown("# 📁 Analizador de CSV con ML\nSube un CSV y entrena un modelo automáticamente.")
    with gr.Row():
        with gr.Column():
            archivo_inp    = gr.File(label="Archivo CSV", file_types=[".csv"])
            col_objetivo   = gr.Textbox(label="Nombre de la columna objetivo",
                                         placeholder="p.ej. target, label, clase")
            modelo_inp     = gr.Radio(["Random Forest", "Regresión Logística"],
                                       value="Random Forest", label="Modelo")
            btn_analizar   = gr.Button("🚀 Analizar", variant="primary")
        with gr.Column():
            resultado_txt  = gr.Markdown(label="Resultados")
            fig_imp_out    = gr.Plot(label="Importancia de variables")
            fig_dist_out   = gr.Plot(label="Distribución de la variable objetivo")

    btn_analizar.click(
        fn=analizar_csv,
        inputs=[archivo_inp, col_objetivo, modelo_inp],
        outputs=[resultado_txt, fig_imp_out, fig_dist_out]
    )

demo_csv.launch(share=True, quiet=True, inbrowser=False)